Parameter Cell

In [11]:
as_of_date = "2026-08-21"   # must match 00_simulate_sources
batch_id = ""

StatementMeta(, 7bc08443-6b93-4029-95fe-0b9e5d594123, 13, Finished, Available, Finished, False)

Setup

In [12]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/code")

from datetime import date
from bikeshare import control, landing
from bikeshare.config import make_config
from bikeshare.sources.world import BikeShareWorld

spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
cfg = make_config(batch_id=batch_id or None)
print("env:", cfg.env, "| posix:", cfg.files_posix, "| spark:", cfg.files_spark)
control.ensure_control_plane(spark)

StatementMeta(, 7bc08443-6b93-4029-95fe-0b9e5d594123, 14, Finished, Available, Finished, False)

env: fabric | posix: /lakehouse/default/Files | spark: Files


In [13]:
world = BikeShareWorld(cfg.source_world)    # only points at the sources, does not create data
as_of = date.fromisoformat(as_of_date)

print("trips   :", landing.land_trip_files(cfg, world))
print("stations:", landing.extract_stations(spark, cfg, world))
print("weather :", landing.land_weather(spark, cfg, world, as_of))
print("events  :", landing.land_events(cfg, world))

StatementMeta(, 7bc08443-6b93-4029-95fe-0b9e5d594123, 15, Finished, Available, Finished, False)

trips   : ['trips_2026-08-06.csv', 'trips_2026-08-07.csv', 'trips_2026-08-08.csv', 'trips_2026-08-09.csv', 'trips_2026-08-10.csv', 'trips_2026-08-11.csv', 'trips_2026-08-12.csv', 'trips_2026-08-13.csv', 'trips_2026-08-14.csv', 'trips_2026-08-15.csv', 'trips_2026-08-16.csv', 'trips_2026-08-17.csv', 'trips_2026-08-18.csv', 'trips_2026-08-19.csv', 'trips_2026-08-20.csv', 'trips_2026-08-21.csv']
stations: {'rows': 2, 'path': 'Files/landing/stations/batch_id=b003'}
weather : {'rows': 24, 'start': '2026-08-21', 'end': '2026-08-21'}
events  : []


In [14]:
spark.table("audit.watermarks").show(truncate=False)

StatementMeta(, 7bc08443-6b93-4029-95fe-0b9e5d594123, 16, Finished, Available, Finished, False)

+------------------------------------------+--------------------------+--------+--------------------------+
|name                                      |value                     |batch_id|updated_at                |
+------------------------------------------+--------------------------+--------+--------------------------+
|silver.station_versions@bronze_ingested_at|2026-11-12 12:23:33.638156|b001    |2026-11-12 12:32:36.854411|
|silver.weather_hourly@bronze_ingested_at  |2026-11-12 12:23:46.354410|b001    |2026-11-12 12:33:44.576134|
|gold.fact_trip@silver_processed_at        |2026-11-12 12:33:03.707041|b001    |2026-11-12 12:40:33.361799|
|silver.events@bronze_ingested_at          |2026-11-12 12:24:03.745833|b001    |2026-11-12 12:34:24.864068|
|silver.trips@bronze_ingested_at           |2026-11-12 12:23:19.361667|b001    |2026-11-12 12:33:20.025756|
|gold.dim_station@modified_at              |2025-04-09 00:00:00.000000|b001    |2026-11-12 12:39:44.508802|
|stations.modified_at@pendin

In [15]:
import sqlite3, os
world = BikeShareWorld(cfg.source_world)

print("source trip files:", len(os.listdir(world.trips_dir)), "| last:", sorted(os.listdir(world.trips_dir))[-1])
con = sqlite3.connect(world.db_path)
print(con.execute("SELECT station_id, capacity, station_status, modified_at FROM stations WHERE station_id IN ('S001','S004')").fetchall())
con.close()

StatementMeta(, 7bc08443-6b93-4029-95fe-0b9e5d594123, 17, Finished, Available, Finished, False)

source trip files: 21 | last: trips_2026-08-21.csv
[('S001', 40, 'active', '2026-08-13 03:00:00'), ('S004', 24, 'maintenance', '2026-08-21 03:00:00')]
